# Gold Layer – Star Schema & KPIs

This notebook builds the **gold layer** from silver data:
1. **Star schema**: `fact_order_items`, `dim_customer`, `dim_product`, `dim_seller`, `dim_date`
2. **KPIs** computed from the gold tables:
   - KPI 1: Monthly revenue & month-over-month growth
   - KPI 2: Top 3 product categories per state
   - KPI 3: Average delivery delay per seller
   - KPI 4: Repeat-customer rate
   - KPI 5: Cumulative revenue over time
   - KPI 6: Revenue by payment type

## 0 — Setup

In [7]:
from spark_windows import configure_windows_hadoop

configure_windows_hadoop()

In [8]:
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F

spark = (
    SparkSession.builder.master("local[4]")
    .appName("Gold Layer")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

In [9]:
SILVER = "./data/silver"
GOLD = "./data/gold"

## 1 — Read Silver Tables

In [10]:
df_orders = spark.read.parquet(f"{SILVER}/olist_orders_dataset.parquet")
df_customers = spark.read.parquet(f"{SILVER}/olist_customers_dataset.parquet")
df_payments = spark.read.parquet(f"{SILVER}/olist_order_payments_dataset.parquet")
df_products = spark.read.parquet(f"{SILVER}/olist_products_dataset.parquet")
df_sellers = spark.read.parquet(f"{SILVER}/olist_sellers_dataset.parquet")
df_order_items = spark.read.parquet(f"{SILVER}/olist_order_items_dataset.parquet")
df_reviews = spark.read.parquet(f"{SILVER}/olist_order_reviews_dataset.parquet")

print("Silver tables loaded:")
for name, df in [
    ("orders", df_orders), ("customers", df_customers),
    ("payments", df_payments), ("products", df_products),
    ("sellers", df_sellers), ("order_items", df_order_items),
    ("reviews", df_reviews),
]:
    print(f"  {name:15s}  {df.count():>8,} rows")

Silver tables loaded:
  orders             99,418 rows
  customers          99,441 rows
  payments          103,886 rows
  products           32,951 rows
  sellers             3,095 rows
  order_items       112,650 rows
  reviews           101,746 rows


## 2 — Dimension Tables

### dim_customer

In [11]:
dim_customer = (
    df_customers
    .select(
        F.monotonically_increasing_id().alias("customer_sk"),
        "customer_id",
        "customer_unique_id",
        "customer_zip_code_prefix",
        "customer_city",
        "customer_state",
    )
)
dim_customer.show(5)
dim_customer.write.mode("overwrite").parquet(f"{GOLD}/dim_customer.parquet")
print(f"✓ dim_customer: {dim_customer.count():,} rows")

+-----------+--------------------+--------------------+------------------------+--------------+--------------+
|customer_sk|         customer_id|  customer_unique_id|customer_zip_code_prefix| customer_city|customer_state|
+-----------+--------------------+--------------------+------------------------+--------------+--------------+
|          0|0001fd6190edaaf88...|94b11d37cd61cb299...|                   29830|  nova venecia|            es|
|          1|00062b33cb9f6fe97...|f90f55ee274a4ae21...|                    2306|     sao paulo|            sp|
|          2|00066ccbe787a588c...|15090f48004f3b0fc...|                   93525| novo hamburgo|            rs|
|          3|000e943451fc2788c...|d73c3cf4a0922ece1...|                   99460|      colorado|            rs|
|          4|00114026c1b7b52ab...|f4dc0a81a11d3d270...|                   22470|rio de janeiro|            rj|
+-----------+--------------------+--------------------+------------------------+--------------+--------------+
o

### dim_product

In [13]:
dim_product = (
    df_products
    .select(
        F.monotonically_increasing_id().alias("product_sk"),
        "product_id",
        "product_category_name",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm",
    )
)
dim_product.show(5)
dim_product.write.mode("overwrite").parquet(f"{GOLD}/dim_product.parquet")
print(f"✓ dim_product: {dim_product.count():,} rows")

+----------+--------------------+---------------------+----------------+-----------------+-----------------+----------------+
|product_sk|          product_id|product_category_name|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+----------+--------------------+---------------------+----------------+-----------------+-----------------+----------------+
|         0|00088930e925c41fd...|           automotivo|            1225|               55|               10|              26|
|         1|0009406fd7479715e...|      cama_mesa_banho|             300|               45|               15|              35|
|         2|000b8f95fcb9e0096...| utilidades_domest...|             550|               19|               24|              12|
|         3|000d9be29b5207b54...|   relogios_presentes|             250|               22|               11|              15|
|         4|0011c512eb256aa0d...|           automotivo|             100|               16|               15|          

### dim_seller

In [14]:
dim_seller = (
    df_sellers
    .select(
        F.monotonically_increasing_id().alias("seller_sk"),
        "seller_id",
        "seller_zip_code_prefix",
        "seller_city",
        "seller_state",
    )
)
dim_seller.show(5)
dim_seller.write.mode("overwrite").parquet(f"{GOLD}/dim_seller.parquet")
print(f"✓ dim_seller: {dim_seller.count():,} rows")

+---------+--------------------+----------------------+-----------+------------+
|seller_sk|           seller_id|seller_zip_code_prefix|seller_city|seller_state|
+---------+--------------------+----------------------+-----------+------------+
|        0|0015a82c2db000af6...|                  9080|santo andre|          sp|
|        1|001cca7ae9ae17fb1...|                 29156|  cariacica|          es|
|        2|001e6ad469a905060...|                 24754|sao goncalo|          rj|
|        3|002100f778ceb8431...|                 14405|     franca|          sp|
|        4|003554e2dce176b55...|                 74565|    goiania|          go|
+---------+--------------------+----------------------+-----------+------------+
only showing top 5 rows
✓ dim_seller: 3,095 rows


### dim_date
One row per distinct date in the orders dataset.

In [15]:
dates = (
    df_orders
    .select(F.col("order_purchase_timestamp").cast("date").alias("date"))
    .filter(F.col("date").isNotNull())
    .distinct()
)

dim_date = (
    dates
    .select(
        F.col("date").alias("date_key"),
        F.year("date").alias("year"),
        F.month("date").alias("month"),
        F.quarter("date").alias("quarter"),
        F.dayofweek("date").alias("day_of_week"),
        F.dayofmonth("date").alias("day_of_month"),
        F.date_format("date", "yyyy-MM").alias("year_month"),
        F.when(F.dayofweek("date").isin(1, 7), True).otherwise(False).alias("is_weekend"),
    )
    .orderBy("date_key")
)
dim_date.show(10)
dim_date.write.mode("overwrite").parquet(f"{GOLD}/dim_date.parquet")
print(f"✓ dim_date: {dim_date.count():,} rows")

+----------+----+-----+-------+-----------+------------+----------+----------+
|  date_key|year|month|quarter|day_of_week|day_of_month|year_month|is_weekend|
+----------+----+-----+-------+-----------+------------+----------+----------+
|2016-09-04|2016|    9|      3|          1|           4|   2016-09|      true|
|2016-09-05|2016|    9|      3|          2|           5|   2016-09|     false|
|2016-09-13|2016|    9|      3|          3|          13|   2016-09|     false|
|2016-09-15|2016|    9|      3|          5|          15|   2016-09|     false|
|2016-10-02|2016|   10|      4|          1|           2|   2016-10|      true|
|2016-10-03|2016|   10|      4|          2|           3|   2016-10|     false|
|2016-10-04|2016|   10|      4|          3|           4|   2016-10|     false|
|2016-10-05|2016|   10|      4|          4|           5|   2016-10|     false|
|2016-10-06|2016|   10|      4|          5|           6|   2016-10|     false|
|2016-10-07|2016|   10|      4|          6|         

## 3 — Fact Table: `fact_order_items`

Grain: one row per item sold.
Includes price, freight, delivery delay, and foreign keys to all dimensions.

In [16]:
fact_order_items = (
    df_order_items
    .join(df_orders.select(
        "order_id", "customer_id", "order_status",
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
    ), on="order_id", how="inner")
    .join(dim_customer.select("customer_sk", "customer_id"), on="customer_id", how="left")
    .join(dim_product.select("product_sk", "product_id"), on="product_id", how="left")
    .join(dim_seller.select("seller_sk", "seller_id"), on="seller_id", how="left")
    .withColumn("purchase_date", F.col("order_purchase_timestamp").cast("date"))
    .join(dim_date.select(F.col("date_key").alias("purchase_date"), F.col("date_key").alias("date_sk")),
          on="purchase_date", how="left")
    .withColumn(
        "delivery_delay_days",
        F.datediff(
            F.col("order_delivered_customer_date"),
            F.col("order_estimated_delivery_date"),
        ),
    )
    .select(
        "order_id",
        "order_item_id",
        "customer_sk",
        "product_sk",
        "seller_sk",
        "date_sk",
        "order_status",
        "price",
        "freight_value",
        "delivery_delay_days",
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
    )
)

fact_order_items.show(5)
fact_order_items.write.mode("overwrite").parquet(f"{GOLD}/fact_order_items.parquet")
print(f"✓ fact_order_items: {fact_order_items.count():,} rows")

+--------------------+-------------+-----------+----------+---------+----------+------------+-----+-------------+-------------------+------------------------+-----------------------------+-----------------------------+
|            order_id|order_item_id|customer_sk|product_sk|seller_sk|   date_sk|order_status|price|freight_value|delivery_delay_days|order_purchase_timestamp|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+-------------+-----------+----------+---------+----------+------------+-----+-------------+-------------------+------------------------+-----------------------------+-----------------------------+
|000229ec398224ef6...|            1|17179886343|     12929|     1117|2018-01-14|   delivered|199.0|        17.87|                -14|     2018-01-14 14:33:31|          2018-01-22 13:19:16|          2018-02-05 00:00:00|
|00042b26cf59d7ce6...|            1|       8532|     11065|     2697|2017-02-04|   delivered|199.9|        18.14|           

In [28]:
df_orders.show()

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|000229ec398224ef6...|6489ae5e4333f3693...|   delivered|     2018-01-14 14:33:31|2018-01-14 14:48:30|         2018-01-16 12:36:48|          2018-01-22 13:19:16|          2018-02-05 00:00:00|
|00042b26cf59d7ce6...|58dbd0b2d70206bf4...|   delivered|     2017-02-04 13:57:51|2017-02-04 14:10:13|         2017-02-16 09:46:09|          2017-03-01 16:42:31|          2017-03-17 00:00:00|
|0005f50442cb953dc...|351d3cb2cee3c7fd0...|  

In [30]:
fact_order_items.show()

+--------------------+-------------+-----------+----------+---------+----------+------------+------+-------------+-------------------+------------------------+-----------------------------+-----------------------------+
|            order_id|order_item_id|customer_sk|product_sk|seller_sk|   date_sk|order_status| price|freight_value|delivery_delay_days|order_purchase_timestamp|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+-------------+-----------+----------+---------+----------+------------+------+-------------+-------------------+------------------------+-----------------------------+-----------------------------+
|000229ec398224ef6...|            1|17179886343|     12929|     1117|2018-01-14|   delivered| 199.0|        17.87|                -14|     2018-01-14 14:33:31|          2018-01-22 13:19:16|          2018-02-05 00:00:00|
|00042b26cf59d7ce6...|            1|       8532|     11065|     2697|2017-02-04|   delivered| 199.9|        18.14|      

---
## 4 — KPIs

All KPIs below are computed from the gold-layer tables only.

### KPI 1: Monthly Revenue & Month-over-Month Growth

In [17]:
# Join delivered orders with payments to get revenue per month
orders_delivered = (
    df_orders
    .filter(
        (F.col("order_status") == "delivered")
        & F.col("order_purchase_timestamp").isNotNull()
    )
    .select("order_id", "order_purchase_timestamp")
)

payments_agg = (
    df_payments
    .groupBy("order_id")
    .agg(F.sum("payment_value").alias("payment_value"))
)

orders_with_payments = orders_delivered.join(payments_agg, on="order_id", how="inner")

w_lag = Window.orderBy("month")

kpi_monthly_revenue = (
    orders_with_payments
    .withColumn("month", F.date_format("order_purchase_timestamp", "yyyy-MM"))
    .groupBy("month")
    .agg(F.round(F.sum("payment_value"), 2).alias("revenue"))
    .withColumn("prev_revenue", F.lag("revenue").over(w_lag))
    .withColumn(
        "mom_growth_pct",
        F.round((F.col("revenue") - F.col("prev_revenue")) / F.col("prev_revenue") * 100, 2),
    )
    .orderBy("month")
)

kpi_monthly_revenue.show(30, truncate=False)
kpi_monthly_revenue.write.mode("overwrite").parquet(f"{GOLD}/kpi_monthly_revenue.parquet")
print("✓ KPI 1: Monthly revenue saved")

+-------+----------+------------+--------------+
|month  |revenue   |prev_revenue|mom_growth_pct|
+-------+----------+------------+--------------+
|2016-10|46566.71  |NULL        |NULL          |
|2016-12|19.62     |46566.71    |-99.96        |
|2017-01|127430.74 |19.62       |649394.09     |
|2017-02|269458.98 |127430.74   |111.46        |
|2017-03|414369.39 |269458.98   |53.78         |
|2017-04|390952.18 |414369.39   |-5.65         |
|2017-05|566872.73 |390952.18   |45.0          |
|2017-06|490225.6  |566872.73   |-13.52        |
|2017-07|566403.93 |490225.6    |15.54         |
|2017-08|646000.61 |566403.93   |14.05         |
|2017-09|700976.01 |646000.61   |8.51          |
|2017-10|751140.27 |700976.01   |7.16          |
|2017-11|1153393.22|751140.27   |53.55         |
|2017-12|843199.17 |1153393.22  |-26.89        |
|2018-01|1078606.86|843199.17   |27.92         |
|2018-02|966510.88 |1078606.86  |-10.39        |
|2018-03|1120678.0 |966510.88   |15.95         |
|2018-04|1132933.95|

### KPI 2: Top 3 Product Categories per State

In [18]:
# Join fact with dimensions to get state + category
items_with_dims = (
    fact_order_items
    .join(dim_customer.select("customer_sk", "customer_state"), on="customer_sk", how="inner")
    .join(dim_product.select("product_sk", "product_category_name"), on="product_sk", how="inner")
)

revenue_by_state_cat = (
    items_with_dims
    .groupBy("customer_state", "product_category_name")
    .agg(F.round(F.sum("price"), 2).alias("total_revenue"))
)

w_rank = Window.partitionBy("customer_state").orderBy(F.desc("total_revenue"))

kpi_top3_categories = (
    revenue_by_state_cat
    .withColumn("rank", F.row_number().over(w_rank))
    .filter(F.col("rank") <= 3)
    .orderBy("customer_state", "rank")
)

kpi_top3_categories.show(30, truncate=False)
kpi_top3_categories.write.mode("overwrite").parquet(f"{GOLD}/kpi_top3_categories_per_state.parquet")
print("✓ KPI 2: Top 3 categories per state saved")

+--------------+----------------------+-------------+----+
|customer_state|product_category_name |total_revenue|rank|
+--------------+----------------------+-------------+----+
|ac            |esporte_lazer         |1677.46      |1   |
|ac            |relogios_presentes    |1389.6       |2   |
|ac            |beleza_saude          |1386.58      |3   |
|al            |beleza_saude          |12780.26     |1   |
|al            |relogios_presentes    |11683.27     |2   |
|al            |informatica_acessorios|7714.36      |3   |
|am            |beleza_saude          |2776.03      |1   |
|am            |ferramentas_jardim    |1980.88      |2   |
|am            |informatica_acessorios|1960.74      |3   |
|ap            |informatica_acessorios|2149.52      |1   |
|ap            |relogios_presentes    |1639.8       |2   |
|ap            |pcs                   |1437.0       |3   |
|ba            |beleza_saude          |51367.91     |1   |
|ba            |relogios_presentes    |48885.59     |2  

### KPI 3: Average Delivery Delay per Seller

Positive = late delivery, negative = early delivery.

In [19]:
kpi_delivery_delay = (
    fact_order_items
    .filter(F.col("delivery_delay_days").isNotNull())
    .join(dim_seller.select("seller_sk", "seller_id", "seller_city", "seller_state"),
          on="seller_sk", how="inner")
    .groupBy("seller_id", "seller_city", "seller_state")
    .agg(
        F.round(F.avg("delivery_delay_days"), 2).alias("avg_delay_days"),
        F.count("*").alias("total_orders"),
        F.round(F.avg(F.when(F.col("delivery_delay_days") > 0, F.col("delivery_delay_days"))), 2)
         .alias("avg_late_days"),
        F.round(
            F.sum(F.when(F.col("delivery_delay_days") > 0, 1).otherwise(0)) / F.count("*") * 100, 2
        ).alias("late_pct"),
    )
    .orderBy(F.desc("avg_delay_days"))
)

kpi_delivery_delay.show(20, truncate=False)
kpi_delivery_delay.write.mode("overwrite").parquet(f"{GOLD}/kpi_delivery_delay_per_seller.parquet")
print("✓ KPI 3: Delivery delay per seller saved")

+--------------------------------+---------------------+------------+--------------+------------+-------------+--------+
|seller_id                       |seller_city          |seller_state|avg_delay_days|total_orders|avg_late_days|late_pct|
+--------------------------------+---------------------+------------+--------------+------------+-------------+--------+
|df683dfda87bf71ac3fc63063fba369d|farroupilha          |rs          |167.0         |1           |167.0        |100.0   |
|8e670472e453ba34a379331513d6aab1|juiz de fora         |mg          |35.0          |1           |35.0         |100.0   |
|4fb41dff7c50136976d1a5cf004a42e2|feira de santana     |ba          |33.0          |3           |134.0        |33.33   |
|8629a7efec1aab257e58cda559f03ba7|goiania              |go          |33.0          |1           |33.0         |100.0   |
|eebb3372362aa9a46975164bed19a7e7|sao paulo            |sp          |27.0          |4           |62.0         |50.0    |
|e09887ca8c7bf8a4621ce481820414e

### KPI 4: Repeat-Customer Rate

**Important**: `customer_id` changes per order. We must use `customer_unique_id`
to identify the actual person across orders.

In [20]:
orders_per_person = (
    df_orders
    .filter(F.col("order_status") == "delivered")
    .join(df_customers.select("customer_id", "customer_unique_id"), on="customer_id", how="inner")
    .groupBy("customer_unique_id")
    .agg(F.count("order_id").alias("order_count"))
)

# Aggregate customer repeat metrics in a single distributed pass (pure Spark SQL)
kpi_repeat_rate = (
    orders_per_person
    .select(
        F.count("*").cast("double").alias("total_unique_customers"),
        F.sum(F.when(F.col("order_count") == 1, 1).otherwise(0)).cast("double").alias("one_time_customers"),
        F.sum(F.when(F.col("order_count") > 1, 1).otherwise(0)).cast("double").alias("repeat_customers"),
    )
    .withColumn(
        "repeat_rate_pct",
        F.round(F.col("repeat_customers") / F.col("total_unique_customers") * 100, 2),
    )
    .selectExpr(
        "stack(4, "
        "'total_unique_customers', total_unique_customers, "
        "'one_time_customers', one_time_customers, "
        "'repeat_customers', repeat_customers, "
        "'repeat_rate_pct', repeat_rate_pct) as (metric, value)"
    )
)

kpi_repeat_rate.show(truncate=False)

# Also build a distribution of order counts
order_count_dist = (
    orders_per_person
    .groupBy("order_count")
    .agg(F.count("*").alias("num_customers"))
    .orderBy("order_count")
)
order_count_dist.show(20)

kpi_repeat_rate.write.mode("overwrite").parquet(f"{GOLD}/kpi_repeat_customer_rate.parquet")
order_count_dist.write.mode("overwrite").parquet(f"{GOLD}/kpi_order_count_distribution.parquet")
print("✓ KPI 4: Repeat-customer rate saved")

+----------------------+-------+
|metric                |value  |
+----------------------+-------+
|total_unique_customers|93336.0|
|one_time_customers    |90536.0|
|repeat_customers      |2800.0 |
|repeat_rate_pct       |3.0    |
+----------------------+-------+

+-----------+-------------+
|order_count|num_customers|
+-----------+-------------+
|          1|        90536|
|          2|         2572|
|          3|          181|
|          4|           28|
|          5|            9|
|          6|            5|
|          7|            3|
|          9|            1|
|         15|            1|
+-----------+-------------+

✓ KPI 4: Repeat-customer rate saved


### KPI 5: Cumulative Revenue Over Time

In [21]:
w_cumulative = Window.orderBy("month").rowsBetween(Window.unboundedPreceding, Window.currentRow)

kpi_cumulative_revenue = (
    kpi_monthly_revenue
    .select("month", "revenue")
    .withColumn("cumulative_revenue", F.round(F.sum("revenue").over(w_cumulative), 2))
)

kpi_cumulative_revenue.show(30, truncate=False)
kpi_cumulative_revenue.write.mode("overwrite").parquet(f"{GOLD}/kpi_cumulative_revenue.parquet")
print("✓ KPI 5: Cumulative revenue saved")

+-------+----------+------------------+
|month  |revenue   |cumulative_revenue|
+-------+----------+------------------+
|2016-10|46566.71  |46566.71          |
|2016-12|19.62     |46586.33          |
|2017-01|127430.74 |174017.07         |
|2017-02|269458.98 |443476.05         |
|2017-03|414369.39 |857845.44         |
|2017-04|390952.18 |1248797.62        |
|2017-05|566872.73 |1815670.35        |
|2017-06|490225.6  |2305895.95        |
|2017-07|566403.93 |2872299.88        |
|2017-08|646000.61 |3518300.49        |
|2017-09|700976.01 |4219276.5         |
|2017-10|751140.27 |4970416.77        |
|2017-11|1153393.22|6123809.99        |
|2017-12|843199.17 |6967009.16        |
|2018-01|1078606.86|8045616.02        |
|2018-02|966510.88 |9012126.9         |
|2018-03|1120678.0 |1.01328049E7      |
|2018-04|1132933.95|1.126573885E7     |
|2018-05|1128836.69|1.239457554E7     |
|2018-06|1011561.35|1.340613689E7     |
|2018-07|1027383.1 |1.443351999E7     |
|2018-08|985414.28 |1.541893427E7     |


### KPI 6: Revenue by Payment Type

In [22]:
kpi_payment_type = (
    df_payments
    .join(
        df_orders
        .filter(F.col("order_status") == "delivered")
        .select("order_id"),
        on="order_id",
        how="inner",
    )
    .groupBy("payment_type")
    .agg(
        F.round(F.sum("payment_value"), 2).alias("total_revenue"),
        F.count("*").alias("transaction_count"),
        F.round(F.avg("payment_value"), 2).alias("avg_transaction_value"),
        F.round(F.avg("payment_installments"), 1).alias("avg_installments"),
    )
    .withColumn(
        "revenue_share_pct",
        F.round(
            F.col("total_revenue") / F.sum("total_revenue").over(Window.partitionBy()) * 100, 2
        ),
    )
    .orderBy(F.desc("total_revenue"))
)

kpi_payment_type.show(truncate=False)
kpi_payment_type.write.mode("overwrite").parquet(f"{GOLD}/kpi_revenue_by_payment_type.parquet")
print("✓ KPI 6: Revenue by payment type saved")

+------------+-------------+-----------------+---------------------+----------------+-----------------+
|payment_type|total_revenue|transaction_count|avg_transaction_value|avg_installments|revenue_share_pct|
+------------+-------------+-----------------+---------------------+----------------+-----------------+
|credit_card |1.209987622E7|74578            |162.24               |3.5             |78.47            |
|boleto      |2767977.98   |19177            |144.34               |1.0             |17.95            |
|voucher     |343013.19    |5493             |62.45                |1.0             |2.22             |
|debit_card  |208066.88    |1485             |140.11               |1.0             |1.35             |
+------------+-------------+-----------------+---------------------+----------------+-----------------+

✓ KPI 6: Revenue by payment type saved


## 5 — Summary

| # | KPI | Gold Table |
|---|-----|-----------|
| 1 | Monthly revenue & MoM growth | `kpi_monthly_revenue` |
| 2 | Top 3 categories per state | `kpi_top3_categories_per_state` |
| 3 | Avg delivery delay per seller | `kpi_delivery_delay_per_seller` |
| 4 | Repeat-customer rate | `kpi_repeat_customer_rate` |
| 5 | Cumulative revenue over time | `kpi_cumulative_revenue` |
| 6 | Revenue by payment type | `kpi_revenue_by_payment_type` |

In [23]:
print("\n" + "=" * 60)
print("  GOLD LAYER COMPLETE")
print("=" * 60)

import os
gold_files = [f for f in os.listdir(GOLD) if f.endswith(".parquet")]
for f in sorted(gold_files):
    print(f"  📦 {f}")

print(f"\n  Total gold tables: {len(gold_files)}")


  GOLD LAYER COMPLETE
  📦 dim_customer.parquet
  📦 dim_date.parquet
  📦 dim_product.parquet
  📦 dim_seller.parquet
  📦 fact_order_items.parquet
  📦 kpi_cumulative_revenue.parquet
  📦 kpi_delivery_delay_per_seller.parquet
  📦 kpi_monthly_revenue.parquet
  📦 kpi_order_count_distribution.parquet
  📦 kpi_repeat_customer_rate.parquet
  📦 kpi_revenue_by_payment_type.parquet
  📦 kpi_top3_categories_per_state.parquet
  📦 payments_per_month_dataset.parquet

  Total gold tables: 13
